# Tutoral: LightGBM LambdaMART Ranker

**LambdaMART** is the boosted tree version of **LambdaRank**.

Here we provide a tutorial code how to train a lightGBM ranker:
* Main code strcure follows the youtube: [
LightGBM Learning to Rank: Build Query-Document Training Data in Python](https://www.youtube.com/watch?v=itJAfkCHe_k&t=496s).
* Useful medium blogs: [A Practical Guide to LambdaMART in LightGbm](https://medium.datadriveninvestor.com/a-practical-guide-to-lambdamart-in-lightgbm-f16a57864f6).
* The objective for training the in LightGBM ranker is **lambdarank**.
* The validation metric is **NDCG**.

**NOTE**: A public dataset **MSLR-WEB10K** from Miscrosoft is release [here](https://www.microsoft.com/en-us/research/project/mslr/).

In [36]:
import lightgbm as lgb
import numpy as np

# Label (target)

In [5]:
judge = {
    ('q1', 'd1'): 3,
    ('q1', 'd2'): 2,
    ('q1', 'd3'): 0,
    ('q2', 'd1'): 1,
    ('q2', 'd2'): 2,
    ('q2', 'd3'): 3
}

num_clicks = {
    ('q1', 'd1'): 8,
    ('q1', 'd2'): 3,
    ('q1', 'd3'): 1,
    ('q2', 'd1'): 2, 
    ('q2', 'd2'): 2, 
    ('q2', 'd3'): 3,
}

labels = {k: judge[k] + 0.3 * min(num_clicks[k], 10) for k in judge}

print(f"labels: {labels}")

val = labels[('q1', 'd2')]
print(f"y_(q1, d2) = {val:.2f}")

labels: {('q1', 'd1'): 5.4, ('q1', 'd2'): 2.9, ('q1', 'd3'): 0.3, ('q2', 'd1'): 1.6, ('q2', 'd2'): 2.6, ('q2', 'd3'): 3.9}
y_(q1, d2) = 2.90


# Data Feature

In [39]:

data_features = {
    ('q1', 'd1'): [2.2, 120, 0.8],
    ('q1', 'd2'): [1.7, 90, 0.5],
    ('q1', 'd3'): [0.3, 110, 0.2],
    ('q2', 'd1'): [0.9, 70, 0.4],
    ('q2', 'd2'): [1.5, 95, 0.6],
    ('q2', 'd3'): [2.3, 135, 0.9],
}

pairs = sorted(labels.keys())

X = [data_features[k] for k in pairs]
y = [labels[k] for k in pairs]
print(f"X: {X}")
print(f"y: {y}")

y = [int(labels[k]*10) for k in pairs]

max_y = max(y)
if max_y > 30:
    norm_y = [int(val * 30/max_y)  for val in y]
else:
    norm_y = y.copy()
print(f"norm y: {norm_y}")

querys = [k[0] for k in pairs]
group = [querys.count(q) for q in sorted(set(querys))]
print(querys, group)

# training data dataloader for lgb
train_data = lgb.Dataset(
    np.array(X),
    label=norm_y,
    group=group,
    feature_name=['bm25', 'doc_len', 'neural_score'],
    params={'feature_pre_filter': False}
)

print(f"train_data type: {type(train_data)}")

print(f"groups={len(group)} rows={len(y)}")

X: [[2.2, 120, 0.8], [1.7, 90, 0.5], [0.3, 110, 0.2], [0.9, 70, 0.4], [1.5, 95, 0.6], [2.3, 135, 0.9]]
y: [5.4, 2.9, 0.3, 1.6, 2.6, 3.9]
norm y: [30, 16, 1, 8, 14, 21]
['q1', 'q1', 'q1', 'q2', 'q2', 'q2'] [3, 3]
train_data type: <class 'lightgbm.basic.Dataset'>
groups=2 rows=6


In [42]:
params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "seed": 42,
    "num_leaves": 15,
    "learning_rate": 0.2,
    "verbose": -1,
    "min_data_in_leaf": 1, # LGBM in default leaf > 20, so need this config to keep run
    "min_sum_hessian_in_leaf": 0.0,
}

# model = lgb.train(params, train_data, num_boost_round=30)

model = lgb.LGBMRanker(
    objective=params["objective"],
    metric=params["metric"],
    num_leaves=params["num_leaves"],
    learning_rate=params["learning_rate"],
    min_data_in_leaf=params["min_data_in_leaf"],
)
model.fit(X=X, y=norm_y, group=group)

preds = model.predict(X)

print(f"relevance pred: {preds}")

for query in ["q1", "q2"]:
    idx = [i for i, k in enumerate(pairs) if k[0] == query]
    top_idx = max(idx, key=lambda i: preds[i])
    print(f"top_q1 = {pairs[top_idx][1]}")

relevance pred: [ 3.38565063 -2.72048113 -2.96248848 -3.72427505 -2.26111307  3.38565063]
top_q1 = d1
top_q1 = d3


/Users/hhung/miniconda3/envs/lightgbm/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


In [45]:
y_int = [min(4, int(round(v))) for v in y]

print(y_int)

param = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "seed": 42,
    # 'label_gain': [0, 1, 3, 7, 15],
    "label_gain": [i for i in range(max(y_int) + 1)],
    "verbose": -1,
    "min_data_in_leaf": 1,
    "min_sum_hessian_in_leaf": 0.0,
}

# label_gain is to store the gain of each label value. By default, label_gain_[i] = (1 << i) - 1. 
# So the default label gain only works with a maximum label value 31. So, in case your label value exceeds 31, 
# you will have to specify your customized label_gain

train_data = lgb.Dataset(np.array(X), label=y_int, group=group)
model = lgb.train(param, train_data, num_boost_round=30)

preds = model.predict(X)

print(f"relevance pred: {preds}")

idx2 = [i for i, k in enumerate(pairs) if k[0] == 'q1']
top_idx2 = max(idx2, key=lambda i: preds[i])

print(idx2, top_idx2)

print(f"top_q1_lambdarank={pairs[top_idx2][1]}")
print(f"top_q1_lambdarank={pairs[top_idx2]}")

[4, 4, 3, 4, 4, 4]
relevance pred: [ 3.38299988  3.38299979 -3.38299982 -3.38299982 -3.38299982  3.38299988]
[0, 1, 2] 0
top_q1_lambdarank=d1
top_q1_lambdarank=('q1', 'd1')
